# Week 11 — Snapshot Monitoring Dashboard
Monitor IBKR snapshot health: coverage, missing periods, IV trends.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import polars as pl
import duckdb
from datetime import datetime, date
from zoneinfo import ZoneInfo
from optlab_research.ibkr.store import ChainStore
from optlab_research.ibkr.scheduler import is_market_open, next_market_open, SnapshotScheduler

ET = ZoneInfo('America/New_York')
store = ChainStore('../../data/ibkr_chains')
print('Monitoring module loaded')

## 1. Scheduler Status

In [ ]:
now = datetime.now(ET)
print(f'Current time (ET): {now.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Market open: {is_market_open(now)}')
if not is_market_open(now):
    nxt = next_market_open(now)
    delta = nxt - now
    hours = delta.total_seconds() / 3600
    print(f'Next open: {nxt.strftime("%Y-%m-%d %H:%M ET")} ({hours:.1f} hours)')

## 2. Snapshot Manifest Summary

In [ ]:
manifest = store.manifest()
if manifest.is_empty():
    print('No snapshots yet. Run scripts/ibkr_snapshot_daemon.py during market hours.')
else:
    summary = store.summary()
    for k, v in summary.items():
        print(f'{k:25s}: {v}')
    print()
    print('By underlying:')
    by_sym = (
        manifest
        .group_by('underlying')
        .agg([
            pl.count().alias('n_snapshots'),
            pl.col('n_rows').sum().alias('total_quotes'),
            pl.col('date').min().alias('first_date'),
            pl.col('date').max().alias('last_date'),
        ])
        .sort('n_snapshots', descending=True)
    )
    print(by_sym)

## 3. Coverage Heatmap — Snapshots per Hour per Day

In [ ]:
if not manifest.is_empty():
    df = manifest.with_columns([
        pl.col('date').cast(pl.Utf8),
        pl.col('hour').cast(pl.Int32),
    ])
    pivot = df.group_by(['date', 'hour']).agg(pl.count().alias('n'))
    dates = sorted(pivot['date'].unique().to_list())
    hours = list(range(9, 17))

    grid = [[0]*len(hours) for _ in dates]
    for row in pivot.iter_rows(named=True):
        if row['date'] in dates and row['hour'] in hours:
            di = dates.index(row['date'])
            hi = hours.index(row['hour'])
            grid[di][hi] = row['n']

    fig, ax = plt.subplots(figsize=(14, max(3, len(dates)*0.5)))
    im = ax.imshow(grid, cmap='YlGn', aspect='auto', vmin=0)
    ax.set_xticks(range(len(hours)))
    ax.set_xticklabels([f'{h}:00' for h in hours])
    ax.set_yticks(range(len(dates)))
    ax.set_yticklabels(dates)
    plt.colorbar(im, ax=ax, label='Snapshots')
    ax.set_title('Snapshot Coverage Heatmap (snapshots per hour)')
    plt.tight_layout()
    plt.savefig('coverage_heatmap.png', dpi=150)
    print('Saved: coverage_heatmap.png')
else:
    print('No data yet.')

## 4. IV Term Structure — Latest Snapshot

In [ ]:
symbol = 'SPY'
df = store.read(symbol, latest_only=True)
if not df.is_empty():
    calls = df.filter(
        (pl.col('right') == 'C') &
        pl.col('implied_vol').is_not_null() &
        pl.col('implied_vol').gt(0)
    )
    if not calls.is_empty():
        by_exp = (
            calls.group_by('expiration')
            .agg([
                pl.col('implied_vol').mean().alias('avg_iv'),
                pl.col('implied_vol').std().alias('iv_std'),
                pl.count().alias('n'),
            ])
            .sort('expiration')
        )
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(range(len(by_exp)), [v*100 for v in by_exp['avg_iv'].to_list()],
               color='steelblue', alpha=0.8)
        ax.errorbar(
            range(len(by_exp)),
            [v*100 for v in by_exp['avg_iv'].to_list()],
            yerr=[v*100 for v in by_exp['iv_std'].to_list()],
            fmt='none', color='navy', capsize=4
        )
        ax.set_xticks(range(len(by_exp)))
        ax.set_xticklabels(by_exp['expiration'].to_list(), rotation=45)
        ax.set_ylabel('Avg Implied Volatility (%)')
        ax.set_title(f'{symbol} IV Term Structure — Latest Snapshot')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('spy_iv_term_structure.png', dpi=150)
        print('Saved: spy_iv_term_structure.png')
        print(by_exp)
    else:
        print('No call IV data in latest snapshot.')
else:
    print(f'No data for {symbol}. Run daemon first.')

## 5. Missing Period Detection

In [ ]:
if not manifest.is_empty():
    # Find gaps > 10 minutes between snapshots on same day
    df = (
        manifest
        .with_columns(pl.col('snapshot_ts').str.to_datetime())
        .sort(['underlying', 'snapshot_ts'])
    )
    gaps = []
    for sym in df['underlying'].unique().to_list():
        sym_df = df.filter(pl.col('underlying') == sym)
        ts_list = sym_df['snapshot_ts'].to_list()
        for i in range(1, len(ts_list)):
            delta_min = (ts_list[i] - ts_list[i-1]).total_seconds() / 60
            if delta_min > 10:
                gaps.append({
                    'symbol': sym,
                    'gap_from': str(ts_list[i-1]),
                    'gap_to': str(ts_list[i]),
                    'gap_minutes': round(delta_min, 1),
                })
    if gaps:
        print(f'Found {len(gaps)} gaps > 10 minutes:')
        for g in gaps[:10]:
            print(f" {g['symbol']}: {g['gap_from']} → {g['gap_to']} ({g['gap_minutes']} min)")
    else:
        print('No gaps detected. Coverage looks complete.')
else:
    print('No manifest data yet.')

## 6. Windows Task Scheduler Setup
To run the daemon automatically on weekdays at 9:00 AM:

```powershell
# Run this in PowerShell as Administrator
$action = New-ScheduledTaskAction `
  -Execute "C:\Users\Owner\anaconda3\python.exe" `
  -Argument "scripts\ibkr_snapshot_daemon.py --interval 5" `
  -WorkingDirectory "C:\Users\Owner\OneDrive\Desktop\School Work\Grad School Work\Options Club\optlab-research"

$trigger = New-ScheduledTaskTrigger `
  -Weekly `
  -DaysOfWeek Monday,Tuesday,Wednesday,Thursday,Friday `
  -At "9:00AM"

Register-ScheduledTask `
  -TaskName "IBKR Snapshot Daemon" `
  -Action $action `
  -Trigger $trigger `
  -RunLevel Highest
```

The daemon will start at 9:00 AM, wait for market open at 9:30 AM, then snapshot every 5 minutes until 4:00 PM, then sleep until the next trading day.